# CrisisOps v2 — GRPO Training Notebook

Single-click runnable in Google Colab. Run cells top-to-bottom with no manual steps.

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl>=0.29.0 transformers datasets matplotlib --quiet
!pip install -q mp-api pymatgen 2>/dev/null || true  # optional

In [ ]:
# Cell 2: Clone or mount the repo (auto-detect Colab vs. local)
import os, sys
if "google.colab" in str(get_ipython()):
    !git clone https://github.com/YOUR_REPO_URL /content/crisisops 2>/dev/null || print("already cloned")
    os.chdir("/content/crisisops")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
# Cell 3: Calibration check (must pass before training)
from calibration.calibrate import run_calibration
result = run_calibration(n_episodes=20, seed=99)
assert result["status"] == "CALIBRATION PASSED", f"Calibration failed: {result}"
print("✓ Calibration passed")

In [ ]:
# Cell 4: Training (Level 1, 50 episodes for quick demo)
from training.grpo_trainer import train
train(curriculum_level=1, num_episodes=50, output_dir="./demo_run", seed=42)

In [ ]:
# Cell 5: Show training curve
from IPython.display import Image
Image("./demo_run/training_curve.png")

In [ ]:
# Cell 6: Replay best checkpoint
import subprocess
result = subprocess.run(
    ["python", "-m", "baselines.replay", "--seed", "42", "--level", "1", "--verbose"],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
# Cell 7: One-shot Jira demo write (dry_run=True by default)
from deployment.jira_adapter import JiraAdapter
adapter = JiraAdapter(dry_run=True)
test_plan = {
    "plan_summary": "Recovery plan: detected 2 deceptive members, reassigned 3 tasks, client satisfaction maintained at 8.2",
    "risk_items": ["Budget 40% over", "Critical path blocked"],
    "timeline": "2025-05-30"
}
adapter.submit_recovery_plan(test_plan)
print("✓ Would write to Jira board (dry_run=True)")